In [0]:
%sql

CREATE OR REPLACE TEMP VIEW stg_account AS SELECT
    account_id,
    account_name,
    account_type,
    created_at,
    updated_at,
    is_updated,
    is_deleted
FROM bakery_pos_develop.silver.account_account
QUALIFY ROW_NUMBER() OVER(
    PARTITION BY account_id
    ORDER BY silver_processed_at DESC
) = 1

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bakery_pos_develop.gold.dim_account_account (
    account_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    account_id INT,
    account_name STRING,
    account_type STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    is_updated BOOLEAN,
    is_deleted BOOLEAN,
    PRIMARY KEY (account_sk)
)

In [0]:
%sql
MERGE INTO bakery_pos_develop.gold.dim_account_account AS tgt
USING stg_account AS src
ON tgt.account_id = src.account_id

WHEN MATCHED AND src.updated_at > tgt.updated_at THEN
    UPDATE SET
        tgt.account_name = src.account_name,
        tgt.account_type = src.account_type,
        tgt.created_at = src.created_at,
        tgt.updated_at = src.updated_at,
        tgt.is_updated = src.is_updated,
        tgt.is_deleted = src.is_deleted

WHEN NOT MATCHED THEN
    INSERT (
        account_id,
        account_name,
        account_type,
        created_at,
        updated_at,
        is_updated,
        is_deleted
    )
    VALUES (
        src.account_id,
        src.account_name,
        src.account_type,
        src.created_at,
        src.updated_at,
        src.is_updated,
        src.is_deleted
    )